<a href="https://colab.research.google.com/github/BhavyeNijhawan/swarkavach/blob/main/notebooks/01_setup_and_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01. Setup and data

Proves the Colab environment can run this project, then pulls the two real
spoofing datasets the later notebooks need.

**Needs:** no GPU. A GPU makes nothing here faster, so run it on a CPU runtime
and save your GPU quota for notebooks 02, 03 and 05.

**Time:** 3 minutes for the environment check and the test suite. The dataset
downloads are the slow part: about 25 to 50 minutes for ASVspoof 2019 LA
(7.12 GB) and 25 to 55 minutes for In-the-Wild (8.16 GB) on a normal Colab
connection. `SUBSET` cuts that down.

**Writes back:** `data/results/dataset_manifest.json` and a merged
`data/results/provenance.json`, zipped and downloaded at the end.

**Order of business**

1. environment: GPU name, torch version, free disk, RAM
2. the project's own test suite, so a broken clone fails here and not in
   notebook 04
3. optional Google Drive mount, because a Colab session dies and takes
   50 GB of downloaded flac with it
4. ASVspoof 2019 LA and In-the-Wild, with a disk check before either starts
5. a fallback that generates the project's own corpus, so notebooks 03, 04
   and 05 still run if a download fails

In [ ]:
# ---------------------------------------------------------------------------
# Get the repository and install the Colab stack.
# Re-running this cell is safe: if the clone is already on disk it pulls.
# ---------------------------------------------------------------------------

# CHANGE THIS to your own fork before you run anything.
REPO_URL = "https://github.com/BhavyeNijhawan/svarkavach.git"
REPO_DIR = "svk_repo"   # NOT "swarkavach": a directory on sys.path whose name matches the package shadows the real install
BRANCH = "main"

import os
import sys
from pathlib import Path

BASE = Path("/content") if Path("/content").exists() else Path.cwd()
REPO = BASE / REPO_DIR

if (REPO / "pyproject.toml").exists():
    print(f"{REPO} is already cloned, pulling instead")
    !git -C "{REPO}" pull --ff-only
else:
    !git clone --branch {BRANCH} {REPO_URL} "{REPO}"

os.chdir(REPO)
if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))
print("cwd:", os.getcwd())

!pip install -q -r requirements-colab.txt
!pip install -q -e .

import swarkavach
print("swarkavach", swarkavach.__version__, "ready")

## Environment

Nothing here is decoration. The later notebooks branch on all four numbers:
notebook 02 halves its epoch count without a GPU, and this notebook refuses
to start a 7 GB download onto 4 GB of free disk.

In [ ]:
import platform
import shutil
import subprocess
import sys

import numpy as np


def nvidia_smi() -> str:
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
            capture_output=True, text=True, timeout=20)
        return out.stdout.strip() or "no GPU"
    except Exception:
        return "nvidia-smi not available"


def free_gb(path=".") -> float:
    return shutil.disk_usage(path).free / 1e9


def total_ram_gb() -> float:
    try:
        import psutil

        return psutil.virtual_memory().total / 1e9
    except Exception:
        try:
            import os

            return os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9
        except Exception:
            return float("nan")


try:
    import torch

    TORCH_VERSION = torch.__version__
    HAS_GPU = torch.cuda.is_available()
    GPU = torch.cuda.get_device_name(0) if HAS_GPU else "none"
except Exception as exc:
    TORCH_VERSION = f"import failed: {exc}"
    HAS_GPU, GPU = False, "none"

du = shutil.disk_usage(".")
rows = [
    ("python", platform.python_version()),
    ("platform", platform.platform()),
    ("numpy", np.__version__),
    ("torch", TORCH_VERSION),
    ("GPU", GPU),
    ("nvidia-smi", nvidia_smi()),
    ("RAM", f"{total_ram_gb():.1f} GB"),
    ("disk total", f"{du.total / 1e9:.1f} GB"),
    ("disk free", f"{du.free / 1e9:.1f} GB"),
]
width = max(len(k) for k, _ in rows)
for k, v in rows:
    print(f"{k:<{width}}  {v}")

if not HAS_GPU:
    print("\nNo GPU on this runtime. Fine for this notebook. "
          "Runtime -> Change runtime type -> T4 GPU before notebook 02.")

## Does the clone actually work

Both test files run as plain scripts, no pytest needed. `test_dsp.py` checks
the hand-written signal processing against values that can be worked out by
hand (a square wave's zero crossings, the STFT against `numpy.fft`, the G.711
round-trip SNR the standard promises). `test_integration.py` generates a small
corpus in a temporary directory and drives the whole pipeline through it.

If either fails, stop. Everything downstream inherits the problem.

In [ ]:
import subprocess
import sys

TEST_RESULTS = {}
for name in ("tests/test_dsp.py", "tests/test_integration.py"):
    print(f"\n{'=' * 70}\n{name}\n{'=' * 70}")
    proc = subprocess.run([sys.executable, name], capture_output=True, text=True)
    tail = (proc.stdout or "").strip().splitlines()
    for line in tail[-30:]:
        print(line)
    if proc.returncode != 0:
        print("--- stderr ---")
        print((proc.stderr or "").strip()[-2000:])
    TEST_RESULTS[name] = int(proc.returncode)
    print(f"[{name}] exit code {proc.returncode}")

if any(TEST_RESULTS.values()):
    print("\nAt least one test file failed. Fix that before downloading 15 GB.")
else:
    print("\nBoth test files passed.")

## What gets downloaded, and the honest size of it

**ASVspoof 2019 Logical Access.** The standard benchmark for text-to-speech
and voice-conversion spoofing detection. Edinburgh DataShare serves the whole
Logical Access partition as one 7.12 GB zip, `LA.zip`, which unpacks to about
23 GB of 16 kHz flac. Inside it are three speaker-disjoint partitions (train,
dev, eval) and the CM protocol files that give bonafide or spoof plus the
attack id A01 to A19 per utterance. Notebook 02 needs the protocols and at
minimum the train and dev flac.

**In-the-Wild.** 31,779 clips of 58 politicians and celebrities scraped from
public video and audio, 20.8 hours genuine and 17.2 hours deepfaked, released
with the paper "Does Audio Deepfake Detection Generalize?" (Muller et al.,
Interspeech 2022). It exists precisely because ASVspoof-trained detectors
collapse on it. Notebook 02 uses it as a held-out generalisation test, and the
gap between the two EERs is a result, not a bug.

**SUBSET** controls how much of that you actually pull:

| value | ASVspoof | In-the-Wild | disk after extract |
| --- | --- | --- | --- |
| `"full"` | download 7.12 GB, extract everything | download 8.16 GB, extract everything | about 45 GB |
| `"dev"` | download 7.12 GB, extract protocols + train + dev only | download, extract a balanced 4000-clip sample | about 20 GB |
| `"kaggle"` | Kaggle mirror, needs `kaggle.json` | Kaggle mirror | varies |
| `"none"` | skip | skip | 0, corpus fallback only |

Note the honest bit in the `"dev"` row: DataShare serves one archive, so you
still download all 7.12 GB. `"dev"` saves extracted disk, not bandwidth. If
bandwidth is the constraint, use `"kaggle"`, whose mirrors are split by
partition, or `"none"` and rely on the generated corpus.

Registration: ASVspoof 2019 on DataShare is an open download under its own
licence, no login. Kaggle needs an API token (Account -> Settings -> Create
New API Token, which gives you `kaggle.json`).

In [ ]:
# ---------------------------------------------------------------------------
# Configuration. Everything you might want to change lives here.
# ---------------------------------------------------------------------------

SUBSET = "dev"          # "full" | "dev" | "kaggle" | "none"
USE_DRIVE = False       # mount Google Drive and keep the data there
FORCE_REDOWNLOAD = False

# Real URLs, checked against the DataShare and Hugging Face listings.
ASVSPOOF_LA_URL = (
    "https://datashare.ed.ac.uk/bitstreams/"
    "a9f87c35-f055-4015-80e2-2fdff0d46269/download"          # LA.zip, 7.12 GB
)
ASVSPOOF_LANDING = "https://datashare.ed.ac.uk/handle/10283/3336"
ASVSPOOF_README_URL = (
    "https://datashare.ed.ac.uk/bitstreams/"
    "765597d6-633f-418a-96b0-39274a8d56c9/download"          # README.txt, 13 kB
)

IN_THE_WILD_URL = (
    "https://huggingface.co/datasets/mueller91/In-The-Wild/"
    "resolve/main/release_in_the_wild.zip"                   # 8.16 GB
)
IN_THE_WILD_LANDING = "https://deepfake-total.com/in_the_wild"

# Kaggle mirrors. Slugs move, so check them on kaggle.com before relying on
# these. The ASVspoof one has a typo in the slug that is part of the slug.
KAGGLE_ASVSPOOF = "awsaf49/asvpoof-2019-dataset"
KAGGLE_IN_THE_WILD = "abdallamohamed312/in-the-wild-dataset"

# How much of In-the-Wild to keep in the "dev" and "kaggle" paths.
ITW_SAMPLE_CLIPS = 4000

print(f"SUBSET={SUBSET}  USE_DRIVE={USE_DRIVE}")
print(f"ASVspoof LA:  {ASVSPOOF_LA_URL}")
print(f"In-the-Wild:  {IN_THE_WILD_URL}")

## Google Drive, optional

A Colab session is deleted when it disconnects, and it disconnects. If you
plan to run notebook 02 more than once, put the datasets on Drive so the
second run starts at training instead of at a 7 GB download.

The catch: free Drive is 15 GB, which does not hold the full extract. With
Drive on, this notebook keeps only the protocols and the dev partition there
regardless of `SUBSET`, and puts anything larger on the session disk.

In [ ]:
from pathlib import Path

DRIVE_ROOT = None
if USE_DRIVE:
    try:
        from google.colab import drive

        drive.mount("/content/drive")
        DRIVE_ROOT = Path("/content/drive/MyDrive/swarkavach")
        DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
        import shutil as _sh

        free = _sh.disk_usage(DRIVE_ROOT).free / 1e9
        print(f"Drive mounted at {DRIVE_ROOT}, {free:.1f} GB free")
    except Exception as exc:
        print(f"Drive mount failed ({type(exc).__name__}: {exc}), staying on session disk")
        DRIVE_ROOT = None
else:
    print("Drive not mounted. Everything lands on the session disk and dies with it.")

# Session disk is where the bulk goes either way.
RAW = Path("/content/raw") if Path("/content").exists() else Path("data/raw")
RAW.mkdir(parents=True, exist_ok=True)
LA_DIR = RAW / "asvspoof2019_la"
ITW_DIR = RAW / "in_the_wild"
print("raw data root:", RAW)

## Measure before you download

The sizes above came from the dataset landing pages, so they are right until
somebody re-uploads. Rather than trusting a constant, ask the server: an HTTP
HEAD gives the exact `content-length` in a second and costs nothing. Then
compare against free disk and decide, out loud, what to do.

In [ ]:
import shutil
import urllib.request


def remote_size_gb(url: str, timeout: int = 30):
    """content-length in GB, or None if the server will not say."""
    req = urllib.request.Request(url, method="HEAD")
    req.add_header("User-Agent", "swarkavach-colab/1.0")
    try:
        with urllib.request.urlopen(req, timeout=timeout) as resp:
            n = resp.headers.get("content-length")
            return float(n) / 1e9 if n else None
    except Exception as exc:
        print(f"  HEAD failed for {url[:60]}...  {type(exc).__name__}: {exc}")
        return None


free = shutil.disk_usage(".").free / 1e9
print(f"free disk: {free:.1f} GB\n")

sizes = {}
for label, url in (("ASVspoof LA.zip", ASVSPOOF_LA_URL),
                   ("In-the-Wild zip", IN_THE_WILD_URL)):
    gb = remote_size_gb(url)
    sizes[label] = gb
    print(f"{label:<18} {gb:.2f} GB" if gb else f"{label:<18} size unknown")

# Extraction needs roughly 3x the zip for flac and 1x for wav, plus the zip
# itself until it is deleted. Budget generously and say so.
need = 0.0
if SUBSET in ("full", "dev"):
    la = sizes.get("ASVspoof LA.zip") or 7.12
    itw = sizes.get("In-the-Wild zip") or 8.16
    need = (la * 4.0 if SUBSET == "full" else la * 1.8) + (itw * 2.0 if SUBSET == "full" else itw * 1.3)

print(f"\nrough space needed for SUBSET={SUBSET!r}: {need:.1f} GB")
if need and need > free * 0.92:
    print("NOT ENOUGH DISK. Falling back to SUBSET='none' and the generated corpus.")
    print("If you want the real data, use a Colab runtime with more disk, "
          "or SUBSET='kaggle' and pull one partition at a time.")
    SUBSET = "none"
else:
    print("Enough disk. Proceeding.")

## ASVspoof 2019 LA

`wget -c` resumes a partial file, which matters because a 7 GB download over
a Colab session will sometimes stall. Re-run the cell and it picks up where it
stopped instead of starting again.

After the download the zip is tested (`unzip -t` reads every entry's CRC), then
extracted. In the `"dev"` path only the protocol directory and the train and
dev flac come out, which is what notebook 02 trains on. The eval partition is
extracted too when there is room, because the headline EER has to be measured
on eval and nothing else.

In [ ]:
import shutil
import subprocess
import time
from pathlib import Path

LA_STATUS = {"requested": SUBSET, "downloaded": False, "root": None, "note": ""}


def run(cmd: str, quiet: bool = False) -> int:
    proc = subprocess.run(cmd, shell=True, capture_output=quiet, text=True)
    return proc.returncode


if SUBSET in ("full", "dev"):
    LA_DIR.mkdir(parents=True, exist_ok=True)
    zip_path = LA_DIR / "LA.zip"
    t0 = time.time()

    if zip_path.exists() and not FORCE_REDOWNLOAD:
        print(f"{zip_path} already here ({zip_path.stat().st_size / 1e9:.2f} GB), "
              f"resuming if incomplete")
    print(f"downloading from {ASVSPOOF_LA_URL}")
    rc = run(f'wget -c -q --show-progress -O "{zip_path}" "{ASVSPOOF_LA_URL}"')
    on_disk = f"{zip_path.stat().st_size / 1e9:.2f} GB" if zip_path.exists() else "no file"
    print(f"wget exit {rc}, {time.time() - t0:.0f}s, {on_disk} on disk")

    if zip_path.exists() and zip_path.stat().st_size > 1e9:
        print("\nverifying the archive (CRC of every entry, takes a couple of minutes)")
        rc = run(f'unzip -t -q "{zip_path}"')
        if rc != 0:
            print("ARCHIVE IS CORRUPT. Delete it and re-run this cell.")
            LA_STATUS["note"] = "zip failed unzip -t"
        else:
            print("archive verified")
            if SUBSET == "full":
                patterns = ""
            else:
                # Protocols plus train and dev flac. Eval is added below when
                # there is room, because the headline EER is an eval number.
                patterns = ('"LA/ASVspoof2019_LA_cm_protocols/*" '
                            '"LA/ASVspoof2019_LA_train/*" '
                            '"LA/ASVspoof2019_LA_dev/*" '
                            '"LA/ASVspoof2019_LA_asv_protocols/*"')
            print(f"\nextracting{' everything' if not patterns else ' protocols, train, dev'}")
            run(f'unzip -n -q "{zip_path}" {patterns} -d "{LA_DIR}"')

            if SUBSET == "dev":
                free_now = shutil.disk_usage(LA_DIR).free / 1e9
                if free_now > 12.0:
                    print(f"{free_now:.1f} GB still free, adding the eval partition")
                    run(f'unzip -n -q "{zip_path}" "LA/ASVspoof2019_LA_eval/*" -d "{LA_DIR}"')
                else:
                    print(f"only {free_now:.1f} GB free, skipping the eval flac. "
                          f"Notebook 02 will report EER on dev and say so.")

            LA_STATUS["downloaded"] = True
            LA_STATUS["root"] = str(LA_DIR / "LA")
    else:
        LA_STATUS["note"] = "download did not complete"

elif SUBSET == "kaggle":
    print("Kaggle path. Upload kaggle.json when prompted.")
    try:
        from google.colab import files as _f

        _f.upload()
        run("mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json")
        run("pip install -q kaggle")
        LA_DIR.mkdir(parents=True, exist_ok=True)
        rc = run(f'kaggle datasets download -d {KAGGLE_ASVSPOOF} -p "{LA_DIR}" --unzip')
        LA_STATUS["downloaded"] = rc == 0
        LA_STATUS["root"] = str(LA_DIR)
        LA_STATUS["note"] = f"kaggle mirror {KAGGLE_ASVSPOOF}"
        if rc != 0:
            print(f"kaggle download failed. Check the slug at "
                  f"https://www.kaggle.com/datasets/{KAGGLE_ASVSPOOF}")
    except Exception as exc:
        print(f"Kaggle path failed: {type(exc).__name__}: {exc}")
        LA_STATUS["note"] = f"kaggle failed: {exc}"

else:
    print("SUBSET='none', skipping ASVspoof. Notebook 02 will refuse to train "
          "on real data and say why.")
    LA_STATUS["note"] = "skipped by SUBSET"

print("\nLA status:", LA_STATUS)

## Verify what landed

Counting files is not verification. The protocol files are the ground truth,
so parse them, count bonafide against spoof, list the attack ids, and check
that every utterance the protocol names has a flac on disk. A protocol row
without its audio is the failure mode that makes notebook 02 train on a
silently smaller set.

In [ ]:
from collections import Counter
from pathlib import Path

PROTOCOL_FILES = {
    "train": "ASVspoof2019.LA.cm.train.trn.txt",
    "dev": "ASVspoof2019.LA.cm.dev.trl.txt",
    "eval": "ASVspoof2019.LA.cm.eval.trl.txt",
}


def find_la_root(base: Path):
    """Locate the directory that holds ASVspoof2019_LA_cm_protocols."""
    if not base.exists():
        return None
    for cand in [base, base / "LA"] + sorted(base.glob("**/ASVspoof2019_LA_cm_protocols")):
        root = cand.parent if cand.name == "ASVspoof2019_LA_cm_protocols" else cand
        if (root / "ASVspoof2019_LA_cm_protocols").is_dir():
            return root
    return None


def parse_protocol(path: Path):
    """Rows of (speaker_id, utt_id, attack_id, label).

    Protocol line format is: SPEAKER UTT - SYSTEM KEY, for example
        LA_0079 LA_T_1138215 - - bonafide
        LA_0079 LA_T_1271820 - A01 spoof
    """
    rows = []
    for line in path.read_text(encoding="utf-8").splitlines():
        parts = line.split()
        if len(parts) < 5:
            continue
        rows.append((parts[0], parts[1], parts[3], parts[4]))
    return rows


LA_ROOT = find_la_root(LA_DIR)
LA_SUMMARY = {}

if LA_ROOT is None:
    print("no ASVspoof protocols found on disk")
else:
    print(f"LA root: {LA_ROOT}\n")
    proto_dir = LA_ROOT / "ASVspoof2019_LA_cm_protocols"
    speaker_sets = {}
    for part, fname in PROTOCOL_FILES.items():
        p = proto_dir / fname
        if not p.exists():
            print(f"{part:<6} protocol missing")
            continue
        rows = parse_protocol(p)
        flac_dir = LA_ROOT / f"ASVspoof2019_LA_{part}" / "flac"
        on_disk = 0
        if flac_dir.is_dir():
            present = {f.stem for f in flac_dir.glob("*.flac")}
            on_disk = sum(1 for _, utt, _, _ in rows if utt in present)
        labels = Counter(r[3] for r in rows)
        attacks = sorted({r[2] for r in rows if r[2] != "-"})
        speakers = {r[0] for r in rows}
        speaker_sets[part] = speakers
        LA_SUMMARY[part] = {
            "utterances": len(rows),
            "bonafide": labels.get("bonafide", 0),
            "spoof": labels.get("spoof", 0),
            "attacks": attacks,
            "speakers": len(speakers),
            "flac_on_disk": on_disk,
        }
        print(f"{part:<6} {len(rows):>6} rows  "
              f"bonafide {labels.get('bonafide', 0):>5}  spoof {labels.get('spoof', 0):>6}  "
              f"speakers {len(speakers):>3}  flac on disk {on_disk:>6}")
        print(f"       attacks: {', '.join(attacks) if attacks else 'none'}")

    # The whole point of the official partitions: no speaker crosses a boundary.
    parts = sorted(speaker_sets)
    for i, a in enumerate(parts):
        for b in parts[i + 1:]:
            overlap = speaker_sets[a] & speaker_sets[b]
            print(f"\nspeaker overlap {a}/{b}: {len(overlap)} "
                  f"({'clean, as documented' if not overlap else 'UNEXPECTED'})")

## In-the-Wild

Same shape of download. The archive holds numbered wav files and one metadata
csv naming the speaker and the bona-fide or spoof label per file. Rather than
hard-coding column names that may change, the cell finds the csv and sniffs
which column carries the label.

In [ ]:
import csv
import io
import subprocess
import time
import zipfile
from collections import Counter
from pathlib import Path

ITW_STATUS = {"downloaded": False, "root": None, "note": ""}

if SUBSET in ("full", "dev"):
    ITW_DIR.mkdir(parents=True, exist_ok=True)
    zip_path = ITW_DIR / "release_in_the_wild.zip"
    t0 = time.time()
    print(f"downloading from {IN_THE_WILD_URL}")
    rc = subprocess.run(
        f'wget -c -q --show-progress -O "{zip_path}" "{IN_THE_WILD_URL}"',
        shell=True).returncode
    print(f"wget exit {rc}, {time.time() - t0:.0f}s")

    if zip_path.exists() and zip_path.stat().st_size > 1e8:
        print(f"{zip_path.stat().st_size / 1e9:.2f} GB on disk, verifying")
        rc = subprocess.run(f'unzip -t -q "{zip_path}"', shell=True).returncode
        if rc != 0:
            print("ARCHIVE IS CORRUPT. Delete it and re-run.")
            ITW_STATUS["note"] = "zip failed unzip -t"
        elif SUBSET == "full":
            subprocess.run(f'unzip -n -q "{zip_path}" -d "{ITW_DIR}"', shell=True)
            ITW_STATUS["downloaded"] = True
        else:
            # Balanced sample: take the metadata, pick an equal number of
            # bona-fide and spoof entries, extract only those wavs. A balanced
            # subset keeps the EER comparable to the published full-set number;
            # an unbalanced one does not.
            with zipfile.ZipFile(zip_path) as z:
                meta_name = next((n for n in z.namelist() if n.lower().endswith(".csv")), None)
                if meta_name is None:
                    print("no csv in the archive, extracting everything instead")
                    z.extractall(ITW_DIR)
                else:
                    rows = list(csv.DictReader(
                        io.TextIOWrapper(z.open(meta_name), encoding="utf-8")))
                    fields = rows[0].keys()
                    lab_col = next(c for c in fields if "label" in c.lower())
                    file_col = next(c for c in fields if "file" in c.lower())
                    by_label = {}
                    for r in rows:
                        by_label.setdefault(r[lab_col].strip().lower(), []).append(r[file_col])
                    per = max(ITW_SAMPLE_CLIPS // max(len(by_label), 1), 1)
                    wanted = []
                    for lab, files in by_label.items():
                        wanted.extend(files[:per])
                        print(f"  {lab}: taking {min(per, len(files))} of {len(files)}")
                    prefix = meta_name.rsplit("/", 1)[0] + "/" if "/" in meta_name else ""
                    z.extract(meta_name, ITW_DIR)
                    got = 0
                    for name in z.namelist():
                        base = name.rsplit("/", 1)[-1]
                        if base in set(wanted):
                            z.extract(name, ITW_DIR)
                            got += 1
                    print(f"  extracted {got} wav files")
                    ITW_STATUS["note"] = f"balanced sample, {got} clips"
            ITW_STATUS["downloaded"] = True
        ITW_STATUS["root"] = str(ITW_DIR)
    else:
        ITW_STATUS["note"] = "download did not complete"

elif SUBSET == "kaggle":
    ITW_DIR.mkdir(parents=True, exist_ok=True)
    rc = subprocess.run(
        f'kaggle datasets download -d {KAGGLE_IN_THE_WILD} -p "{ITW_DIR}" --unzip',
        shell=True).returncode
    ITW_STATUS["downloaded"] = rc == 0
    ITW_STATUS["root"] = str(ITW_DIR)
    ITW_STATUS["note"] = f"kaggle mirror {KAGGLE_IN_THE_WILD}"
    if rc != 0:
        print(f"check the slug at https://www.kaggle.com/datasets/{KAGGLE_IN_THE_WILD}")

else:
    print("SUBSET='none', skipping In-the-Wild. Notebook 02 will report "
          "ASVspoof numbers only and mark the generalisation column as not run.")
    ITW_STATUS["note"] = "skipped by SUBSET"

print("\nIn-the-Wild status:", ITW_STATUS)

In [ ]:
import csv
from collections import Counter
from pathlib import Path

ITW_SUMMARY = {}
meta_files = sorted(ITW_DIR.glob("**/*.csv")) if ITW_DIR.exists() else []

if not meta_files:
    print("no In-the-Wild metadata on disk")
else:
    meta = meta_files[0]
    rows = list(csv.DictReader(meta.open(encoding="utf-8")))
    lab_col = next(c for c in rows[0] if "label" in c.lower())
    file_col = next(c for c in rows[0] if "file" in c.lower())
    spk_col = next((c for c in rows[0] if "speaker" in c.lower()), None)

    wavs = {p.name: p for p in ITW_DIR.glob("**/*.wav")}
    present = [r for r in rows if r[file_col] in wavs]
    labels = Counter(r[lab_col].strip().lower() for r in present)
    speakers = {r[spk_col] for r in present} if spk_col else set()

    ITW_SUMMARY = {
        "metadata": str(meta),
        "rows_in_metadata": len(rows),
        "wav_on_disk": len(wavs),
        "usable": len(present),
        "labels": dict(labels),
        "speakers": len(speakers),
    }
    print(f"metadata      {meta}")
    print(f"metadata rows {len(rows)}")
    print(f"wav on disk   {len(wavs)}")
    print(f"usable        {len(present)}")
    print(f"labels        {dict(labels)}")
    print(f"speakers      {len(speakers)}")

## Fallback: the project's own corpus

This is not a consolation prize. The generated corpus is what carries the
Hinglish text branch, the four-cell evaluation grid and the cross-modal
features, none of which exist in ASVspoof or In-the-Wild. Notebooks 03, 04 and
05 read it whether or not the real datasets downloaded, so generate it here
either way and let the download be the optional part.

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

N_CALLS = 480
print(f"generating {N_CALLS} calls with audio, a couple of minutes")
proc = subprocess.run(
    [sys.executable, "-m", "swarkavach.cli", "gen-corpus",
     "--n", str(N_CALLS), "--audio"],
    capture_output=True, text=True)
print((proc.stdout or "")[-3000:])
if proc.returncode != 0:
    print("--- stderr ---")
    print((proc.stderr or "")[-3000:])

CORPUS_OK = False
try:
    from swarkavach.corpus.generator import corpus_stats, load_corpus

    calls = load_corpus()
    stats = corpus_stats(calls)
    CORPUS_OK = len(calls) > 0
    print(f"\n{len(calls)} calls on disk")
    print("cells:", stats.get("cells"))
    print("splits:", stats.get("splits"))
except Exception as exc:
    print(f"\ncorpus not loadable: {type(exc).__name__}: {exc}")
    print("corpus/generator.py may still be in progress. "
          "Notebooks 03 to 05 need it.")

## What landed

In [ ]:
import shutil
from pathlib import Path


def dir_size_gb(path) -> float:
    path = Path(path)
    if not path.exists():
        return 0.0
    return sum(f.stat().st_size for f in path.rglob("*") if f.is_file()) / 1e9


def count(path, pattern) -> int:
    path = Path(path)
    return sum(1 for _ in path.rglob(pattern)) if path.exists() else 0


rows = [
    ("ASVspoof 2019 LA", LA_DIR, "*.flac", LA_STATUS.get("note", "")),
    ("In-the-Wild", ITW_DIR, "*.wav", ITW_STATUS.get("note", "")),
    ("generated corpus audio", Path("data/corpus/audio"), "*.wav", ""),
    ("generated corpus calls", Path("data/corpus/calls"), "*.json", ""),
]

print(f"{'dataset':<26} {'files':>8} {'size GB':>9}  note")
print("-" * 78)
for label, path, pattern, note in rows:
    print(f"{label:<26} {count(path, pattern):>8} {dir_size_gb(path):>9.2f}  {note}")
print("-" * 78)
print(f"{'free disk':<26} {'':>8} {shutil.disk_usage('.').free / 1e9:>9.2f}")

## Export

`dataset_manifest.json` is what notebooks 02 and 04 read to find the data, and
what the console's provenance table shows so a reader can tell which numbers
came from a full download and which came from a 4000-clip sample.

In [ ]:
# ---------------------------------------------------------------------------
# Export helpers. Each notebook carries its own copy so it can run on its own.
#
# Everything written here lands in data/results/ with "source": "colab", a UTC
# timestamp and the GPU name, which is what the local console's provenance
# table reads to tell your laptop numbers apart from your Colab numbers.
# ---------------------------------------------------------------------------
import json
import platform
import zipfile
from datetime import datetime, timezone
from pathlib import Path

NOTEBOOK = "01_setup_and_data"
RESULTS = Path("data/results")
RESULTS.mkdir(parents=True, exist_ok=True)


def gpu_name() -> str:
    try:
        import torch

        if torch.cuda.is_available():
            return torch.cuda.get_device_name(0)
    except Exception:
        pass
    return "none (CPU runtime)"


def utc_now() -> str:
    return datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")


def colab_stamp(notes: str = "") -> dict:
    try:
        import torch

        tv = str(torch.__version__)
    except Exception:
        tv = "not installed"
    return {
        "source": "colab",
        "generated": utc_now(),
        "gpu": gpu_name(),
        "torch": tv,
        "python": platform.python_version(),
        "notebook": NOTEBOOK,
        "notes": notes,
    }


WRITTEN = []


def write_result(name: str, obj: dict, notes: str = "") -> Path:
    """Write one result file, stamped, and remember it for the zip."""
    doc = dict(obj)
    doc.update(colab_stamp(notes))
    path = RESULTS / name
    path.write_text(json.dumps(doc, indent=2, default=float), encoding="utf-8")
    if name not in WRITTEN:
        WRITTEN.append(name)
    print(f"wrote {path}  ({path.stat().st_size / 1024:.1f} kB)")
    return path


def update_provenance(entries) -> Path:
    """Merge these Colab entries into data/results/provenance.json.

    Entries for other artifacts are kept, so a local `swarkavach evaluate` run
    and a Colab run can sit in the same table. Unzip the Colab results AFTER
    the local evaluate, otherwise the local file overwrites this one.
    """
    path = RESULTS / "provenance.json"
    doc = {"entries": []}
    if path.exists():
        try:
            doc = json.loads(path.read_text(encoding="utf-8"))
        except Exception:
            pass
    new_names = {e["artifact"] for e in entries}
    kept = [e for e in doc.get("entries", []) if e.get("artifact") not in new_names]
    doc["entries"] = kept + list(entries)
    doc["generated"] = utc_now()
    doc["host"] = platform.platform()
    doc["gpu"] = gpu_name()
    path.write_text(json.dumps(doc, indent=2), encoding="utf-8")
    if "provenance.json" not in WRITTEN:
        WRITTEN.append("provenance.json")
    print(f"provenance now lists {len(doc['entries'])} artifacts")
    return path


def export_zip(extra_paths=(), zip_name: str = None) -> Path:
    """Zip the result files plus anything extra, then hand it to the browser."""
    out = Path(zip_name or f"swarkavach_{NOTEBOOK}.zip")
    with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
        for name in WRITTEN:
            f = RESULTS / name
            if f.exists():
                z.write(f, arcname=f"data/results/{f.name}")
        for extra in extra_paths:
            p = Path(extra)
            if p.exists():
                z.write(p, arcname=p.as_posix())
            else:
                print(f"  skipped missing {p}")

    names = zipfile.ZipFile(out).namelist()
    print(f"\n{out.name}  {out.stat().st_size / 1024:.1f} kB")
    for n in names:
        print("   ", n)

    try:
        from google.colab import files

        files.download(str(out))
        print("\nDownload started. Unzip it over your local project root.")
    except Exception as exc:
        print(f"\nnot running on Colab ({type(exc).__name__}), zip is at {out.resolve()}")
    return out

In [ ]:
manifest = {
    "subset": SUBSET,
    "drive_used": bool(DRIVE_ROOT),
    "urls": {
        "asvspoof_la": ASVSPOOF_LA_URL,
        "asvspoof_landing": ASVSPOOF_LANDING,
        "in_the_wild": IN_THE_WILD_URL,
        "in_the_wild_landing": IN_THE_WILD_LANDING,
        "kaggle_asvspoof": KAGGLE_ASVSPOOF,
        "kaggle_in_the_wild": KAGGLE_IN_THE_WILD,
    },
    "asvspoof": {
        "status": LA_STATUS,
        "root": str(LA_ROOT) if LA_ROOT else None,
        "partitions": LA_SUMMARY,
    },
    "in_the_wild": {"status": ITW_STATUS, "summary": ITW_SUMMARY},
    "corpus_generated": bool(CORPUS_OK),
    "tests": TEST_RESULTS,
    "free_disk_gb": round(shutil.disk_usage(".").free / 1e9, 2),
}

write_result("dataset_manifest.json", manifest,
             notes=f"SUBSET={SUBSET}, environment check and dataset download")

update_provenance([{
    "artifact": "dataset_manifest.json",
    "source": "colab",
    "generated": utc_now(),
    "notes": (f"ASVspoof LA {'downloaded' if LA_STATUS['downloaded'] else 'not downloaded'}, "
              f"In-the-Wild {'downloaded' if ITW_STATUS['downloaded'] else 'not downloaded'}, "
              f"SUBSET={SUBSET}"),
}])

export_zip()

## Next

Switch the runtime to a T4 GPU and open `02_antispoof_train.ipynb`. It expects
`data/results/dataset_manifest.json` to exist, or it re-derives the paths from
`/content/raw` on its own.

If the datasets did not download, notebook 02 still runs: it trains RawNetLite
on the generated corpus instead and labels every number as generated-audio, so
nothing in the report claims a real-data result it does not have.